# Team MLB · Logistic Regression 실험

데이터팀이 만든 특징 CSV(`features_*.csv`, 정보 열 6개 + 특징 110개)를 읽어서 학습한다.

| 순서 | 내용 |
| --- | --- |
| 0 | 설정 · 경로 |
| 1 | CSV 로드 · 점검 |
| 2 | 학습/최종평가 분할 (모델팀 공유) |
| 3 | 공통 함수: 표준화+LR, 5-fold로 C 선택 |
| 4 | 실험 ① Closed-world (95개 사이트) + 튜닝 전(C=1) 비교 |
| 5 | 실험 ② Open-world 이진 (mon 1 / unmon -1) |
| 5-추가 | 클래스 가중치 균형 비교 (제안서 외) |
| 6 | 실험 ③ Open-world 다중 (95개 + unmon 1개 = 96 클래스) |
| 7 | 초반 제거 비교 trim0/10/20 (Open-world 이진) |
| 7-추가 | 초반 패킷 방향 비교 (trim 결과 해석용, 원본 pkl 사용) |
| 8 | 계수 해석 (그룹별 기여) |
| 8-추가 | 오류 분석 (사이트별 정확도, 혼동 쌍) |
| 9 | (선택) 특징 그룹 제거 실험 |
| 9-추가 | 그룹 제거 실험 (Open-world 이진) |
| 실험 1 | Feature Group별 단독 학습 (Group-only) |
| 실험 2 | 누적 흐름(g6) 50개 vs 100개 |
| 실험 5 준비 | 공통 fold · OOF/test 확률 저장 (실험 6 앙상블에도 사용) |
| 실험 5 | (보조) Open-world 이진 임계값 분석 |
| 실험 6 | → 별도 노트북 `MLB_Experiment6_Ensemble.ipynb` |
| 10 | 결과 저장 |

## 0. 설정

In [ ]:
import os, sys, subprocess
from pathlib import Path

def find_repo_root():
    """저장소 루트(splits/ 폴더가 있는 위치)를 찾는다. Colab에서 노트북만 연 경우 저장소를 내려받는다."""
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "splits" / "split_seed42.csv").exists():
            return p
    if "google.colab" in sys.modules:
        dst = Path("/content/website-fingerprinting")
        if not dst.exists():
            subprocess.run(["git", "clone", "-q", "https://github.com/EWHA-MLB/website-fingerprinting.git", str(dst)], check=True)
        return dst
    raise FileNotFoundError("저장소 안에서 실행해야 한다 (splits/ 폴더를 찾지 못함)")

ROOT = find_repo_root()
FEATURE_DIR = str(ROOT / "features" / "기본 실험")
RESULT_DIR  = str(ROOT / "results" / "logistic_regression")
SPLIT_PATH  = str(ROOT / "splits" / "split_seed42.csv")
RAW_DIR     = str(ROOT / "data")        # 원본 pkl (7-추가에서만 사용, data/README.md 참고)

UNMON_NAME  = "unmon10k"
TRIM        = 0
TRIMS_TO_COMPARE = [0, 10, 20]  # 7번 실험용

SEED      = 42
TEST_SIZE = 0.2                 # 최종 평가용 비율 (mon은 페이지별 20개 중 4개)
C_GRID    = [0.01, 0.1, 1, 10, 100, 1000]
N_FOLDS   = 5
N_JOBS    = -1
MAX_ITER  = 3000

RUN_ABLATION = True             # 9번 실험. 팀에서 대표 모델을 LR로 정했을 때만 True


In [ ]:
os.makedirs(RESULT_DIR, exist_ok=True)
print("ROOT       :", ROOT)
print("FEATURE_DIR:", FEATURE_DIR)
print("RESULT_DIR :", RESULT_DIR)
print("SPLIT_PATH :", SPLIT_PATH)

In [ ]:
import time, json, glob, pickle
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score

pd.set_option("display.max_columns", 50)

## 1. CSV 로드 · 점검

- 열은 정보 6개(`record_id, dataset, page_id, sample_idx, label, trim`) + 특징 110개(`g1_` ~ `g6_`)여야 한다.
- 공통 규칙 8절에 따라 NaN은 없어야 한다. 있으면 데이터팀에 알린다.

In [ ]:
INFO_COLS = ["record_id", "dataset", "page_id", "sample_idx", "label", "trim"]

def load_features(kind, trim):
    path = os.path.join(FEATURE_DIR, f"features_{kind}_trim{trim}.csv")
    df = pd.read_csv(path)
    feat = [c for c in df.columns if c[:2] in ("g1","g2","g3","g4","g5","g6") and c[2] == "_"]
    assert list(df.columns[:6]) == INFO_COLS, f"정보 열 순서가 다름: {path}"
    assert len(feat) == 110, f"특징 수 {len(feat)}개: {path}"
    n_nan = int(df[feat].isna().sum().sum())
    n_inf = int(np.isinf(df[feat].to_numpy(dtype=float)).sum())
    print(f"{os.path.basename(path)}: {df.shape}, NaN {n_nan}, inf {n_inf}")
    if n_nan or n_inf:
        df[feat] = df[feat].replace([np.inf, -np.inf], np.nan).fillna(0)
        print("  → NaN/inf를 0으로 대체함 (데이터팀에 공유 필요)")
    return df, feat

mon, FEATS = load_features("mon", TRIM)
unmon, _   = load_features(UNMON_NAME, TRIM)
GROUPS = {g: [c for c in FEATS if c.startswith(g + "_")] for g in ["g1","g2","g3","g4","g5","g6"]}
print({g: len(v) for g, v in GROUPS.items()})
print("mon 라벨 범위:", mon.label.min(), "~", mon.label.max(), "/ 사이트당 기록 수:", mon.label.value_counts().unique())

features_mon_trim0.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim0.csv: (10000, 116), NaN 0, inf 0
{'g1': 3, 'g2': 17, 'g3': 6, 'g4': 3, 'g5': 31, 'g6': 50}
mon 라벨 범위: 0 ~ 94 / 사이트당 기록 수: [200]


## 2. 학습 / 최종 평가 분할 (모델팀 공유)

**monitored는 하위 페이지별 비율을 유지**하고, **unmonitored는 기록 단위로 무작위** 분할한다. 모든 모델이 같은 분할을 쓴다.

- 분할은 `record_id` 기준이라 trim0/10/20 파일 모두에 같은 분할이 적용된다.
- 분할 파일이 이미 있으면 그대로 읽고, 없으면 만들어서 저장한다.
- 최종 평가 기록은 C 선택에 쓰지 않는다

In [ ]:
if os.path.exists(SPLIT_PATH):
    split = pd.read_csv(SPLIT_PATH)
    print("기존 분할 파일 사용:", SPLIT_PATH)
else:
    tr_m, te_m = train_test_split(mon.record_id, test_size=TEST_SIZE, random_state=SEED, stratify=mon.page_id)
    tr_u, te_u = train_test_split(unmon.record_id, test_size=TEST_SIZE, random_state=SEED)
    split = pd.concat([
        pd.DataFrame({"record_id": tr_m, "split": "train"}), pd.DataFrame({"record_id": te_m, "split": "test"}),
        pd.DataFrame({"record_id": tr_u, "split": "train"}), pd.DataFrame({"record_id": te_u, "split": "test"}),
    ]).sort_values("record_id").reset_index(drop=True)
    split.to_csv(SPLIT_PATH, index=False)
    print("새 분할 파일 저장:", SPLIT_PATH)

SPLIT = dict(zip(split.record_id, split.split))
def attach_split(df):
    s = df.record_id.map(SPLIT)
    assert s.notna().all(), "분할 파일에 없는 record_id가 있음 (unmon 파일 종류가 다른지 확인)"
    df = df.copy()
    df["split"] = s.to_numpy()
    return df

mon, unmon = attach_split(mon), attach_split(unmon)
_all = pd.concat([mon, unmon], ignore_index=True)
print(pd.crosstab(_all.dataset, _all.split))

기존 분할 파일 사용: /content/drive/MyDrive/MLB-Project/results/split_seed42.csv
split     test  train
dataset              
mon       3800  15200
unmon10k  2000   8000


## 3. 공통 함수

- **표준화 + LR** 을 `Pipeline`으로 묶어서, 5-fold 교차검증의 **각 학습 구간에서만** 표준화 기준(평균·표준편차)을 계산한다.
- `GridSearchCV`로 C를 고른 뒤, 학습 데이터 전체로 다시 학습하고 최종 평가 기록으로 한 번만 평가한다.
- `C_grid=[값 하나]`를 주면 C를 고정한다. 조건끼리 "같은 모델 설정"으로 비교해야 하는 7번·9번 실험에서 쓴다.
- 수렴 경고를 끄지 않고 **반복 횟수(`n_iter`)** 를 기록한다. `MAX_ITER`에 닿으면 경고를 출력한다.
- best C가 탐색 범위의 끝 값이면 경고를 출력한다.
- 탐색 시간·재학습 시간·예측 시간과 C별 교차검증 표(`CV_TABLES`)도 같이 기록한다.

In [ ]:
def make_lr(class_weight=None):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=MAX_ITER, random_state=SEED, class_weight=class_weight))

CV_TABLES = {}   # 실험별 C 탐색 표 (보고서용)

def fit_lr(X_tr, y_tr, name, class_weight=None, C_grid=None):
    """C_grid를 [값 하나]로 주면 탐색 없이 그 C로 고정해서 5-fold 점수만 구한다."""
    C_grid = C_GRID if C_grid is None else C_grid
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    gs = GridSearchCV(make_lr(class_weight), {"logisticregression__C": C_grid}, cv=cv,
                      scoring="accuracy", n_jobs=N_JOBS, refit=True, return_train_score=False)
    t0 = time.time(); gs.fit(X_tr, y_tr); t_fit = time.time() - t0
    cvres = pd.DataFrame(gs.cv_results_)[["param_logisticregression__C", "mean_test_score", "std_test_score", "mean_fit_time"]]
    cvres.columns = ["C", "cv_acc_mean", "cv_acc_std", "fit_time_per_fold"]
    CV_TABLES[name] = cvres.assign(experiment=name)
    n_iter = int(np.max(gs.best_estimator_[-1].n_iter_))
    info = dict(best_C=gs.best_params_["logisticregression__C"], cv_acc=gs.best_score_,
                cv_std=cvres.loc[gs.best_index_, "cv_acc_std"], n_iter=n_iter,
                refit_time_s=gs.refit_time_, search_time_s=t_fit)
    print(f"[{name}] best C = {info['best_C']} | CV acc = {info['cv_acc']:.4f} ± {info['cv_std']:.4f} | 반복 {n_iter}회 | 총 탐색 {t_fit:.1f}s")
    if n_iter >= MAX_ITER:
        print(f"  [경고] max_iter({MAX_ITER})에 도달 → 수렴하지 않았을 수 있음. MAX_ITER를 늘려 다시 실행")
    if len(C_grid) > 1 and info["best_C"] in (min(C_grid), max(C_grid)):
        print("  [경고] best C가 탐색 범위의 끝 값 → 범위를 넓혀 확인 필요")
    display(cvres.round(4))
    return gs.best_estimator_, info

def timed_predict(model, X):
    t0 = time.time(); p = model.predict(X); return p, time.time() - t0

RESULTS = []   # 실험별 요약
PREDS = {}     # 실험별 최종 평가 예측 (오류 분석용)

## 4. 실험 ① Closed-world: 관심 사이트 95개 식별

- 데이터: mon만, 라벨 0~94
- 지표: 정확도 (+ 참고용 Macro-F1)
- **튜닝 전후 비교**(제안서 6-4 ③): C를 고른 결과(`closed_world`)와 기본값 C=1 결과(`closed_world_default_C1`)를 둘 다 남긴다.

In [ ]:
def run_closed(mon, feats, name="closed_world", C_grid=None):
    tr, te = mon[mon.split == "train"], mon[mon.split == "test"]
    model, info = fit_lr(tr[feats], tr.label, name, C_grid=C_grid)
    pred, t_pred = timed_predict(model, te[feats])
    acc = accuracy_score(te.label, pred)
    row = dict(experiment=name, trim=int(mon.trim.iloc[0]), **info, test_acc=acc,
               test_macro_f1=f1_score(te.label, pred, average="macro"), predict_time_s=t_pred)
    print(f"→ test acc {acc:.4f}")
    PREDS[name] = te[["record_id", "label"]].assign(pred=pred)
    return row, model

row, model_closed = run_closed(mon, FEATS)
RESULTS.append(row)
BEST_C_CLOSED = row["best_C"]   # 9번 그룹 제거 실험에서 고정해서 쓴다

# 튜닝 전(기본값 C=1) 성능
row_default, _ = run_closed(mon, FEATS, name="closed_world_default_C1", C_grid=[1.0])
RESULTS.append(row_default)

[closed_world] best C = 100 | CV acc = 0.6626 ± 0.0065 | 반복 1221회 | 총 탐색 1042.5s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,0.01,0.4470,0.0035,13.8017
1,0.10,0.5571,0.0031,20.1793
2,1.00,0.6239,0.0054,23.9565
3,10.00,0.6530,0.0058,40.8906
4,100.00,0.6626,0.0065,84.7311
5,1000.00,0.6613,0.0092,147.8069


→ test acc 0.6884
[closed_world_default_C1] best C = 1.0 | CV acc = 0.6239 ± 0.0054 | 반복 334회 | 총 탐색 112.7s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,1.0,0.6239,0.0054,21.8285


→ test acc 0.6413


## 5. 실험 ② Open-world 이진: 관심 사이트인가?

- 데이터: mon(라벨 1) + unmon(라벨 -1)
- 지표:
  - 정확도 (+ 비교 기준: 전부 다수 범주로 답했을 때의 정확도 `majority_baseline`)
  - **mon → mon 비율**: 실제 관심 사이트 기록 중 관심 사이트로 판단한 비율 (= TPR, recall)
  - **unmon → unmon 비율**: 실제 관심 밖 기록 중 관심 밖으로 판단한 비율 (= TNR)
  - 참고: precision, FPR
  - 참고: ROC-AUC, PR-AUC. 판정 기준값 0.5 한 점이 아니라 mon일 확률 전체로 본 구분 능력이다.
- mon일 확률(`p_mon`)을 예측 파일에 같이 저장해서, 나중에 기준값을 바꿔 보거나 곡선을 그릴 수 있게 한다.

In [ ]:
def binary_view(mon, unmon):
    df = pd.concat([mon, unmon], ignore_index=True)
    df["y"] = np.where(df.dataset == "mon", 1, -1)
    return df

def openworld_metrics(y_true_bin, y_pred_bin):
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[-1, 1]).ravel()
    return dict(mon_as_mon=tp / (tp + fn), unmon_as_unmon=tn / (tn + fp),
                precision=tp / (tp + fp) if tp + fp else 0.0, fpr=fp / (fp + tn),
                TP=tp, FN=fn, TN=tn, FP=fp)

def run_open_binary(mon, unmon, feats, name="open_binary", class_weight=None, C_grid=None):
    df = binary_view(mon, unmon)
    tr, te = df[df.split == "train"], df[df.split == "test"]
    model, info = fit_lr(tr[feats], tr.y, name, class_weight=class_weight, C_grid=C_grid)
    pred, t_pred = timed_predict(model, te[feats])
    p_mon = model.predict_proba(te[feats])[:, list(model.classes_).index(1)]   # mon일 확률
    m = openworld_metrics(te.y, pred)
    row = dict(experiment=name, trim=int(df.trim.iloc[0]), **info,
               test_acc=accuracy_score(te.y, pred), majority_baseline=max((te.y == 1).mean(), (te.y == -1).mean()),
               roc_auc=roc_auc_score(te.y, p_mon), pr_auc=average_precision_score(te.y, p_mon, pos_label=1),
               predict_time_s=t_pred, **m)
    print(f"→ test acc {row['test_acc']:.4f} (전부 다수 범주로 답할 때 {row['majority_baseline']:.4f}) | "
          f"mon→mon {m['mon_as_mon']:.4f} | unmon→unmon {m['unmon_as_unmon']:.4f} | ROC-AUC {row['roc_auc']:.4f}")
    PREDS[f"{name}_trim{row['trim']}"] = te[["record_id", "dataset", "y"]].assign(pred=pred, p_mon=p_mon)
    return row, model

row, model_binary = run_open_binary(mon, unmon, FEATS)
RESULTS.append(row)
BEST_C_BINARY = row["best_C"]   # 7번 초반 제거 비교에서 고정해서 쓴다

[open_binary] best C = 10 | CV acc = 0.7810 ± 0.0042 | 반복 573회 | 총 탐색 104.7s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,0.01,0.7708,0.0029,0.6643
1,0.10,0.7785,0.0045,1.3008
2,1.00,0.7803,0.0037,3.5060
3,10.00,0.7810,0.0042,6.7134
4,100.00,0.7810,0.0047,11.7002
5,1000.00,0.7810,0.0039,12.8107


→ test acc 0.7752 (전부 다수 범주로 답할 때 0.6552) | mon→mon 0.8924 | unmon→unmon 0.5525 | ROC-AUC 0.8119


### 5-추가. (제안서 외 추가 실험) 클래스 가중치 균형 비교

> 제안서에 없는 **추가 비교용 실험**이다. 필수 실험 ②의 기본 결과는 위 결과를 그대로 쓴다.

**왜 하는가**
- 기본 실험 ②에서 mon→mon에 비해 unmon→unmon이 낮게 나온다(unmon을 관심 사이트로 오인).
- 학습 데이터가 mon 15,200개 vs unmon 8,000개로 약 2:1이라, 모델이 애매하면 mon 쪽으로 판단하도록 기울었을 가능성이 있다.

**무엇을 바꾸는가**
- `LogisticRegression(class_weight="balanced")`: 클래스별 기록 수에 반비례하는 가중치를 줘서, 수가 적은 unmon을 틀렸을 때의 손실을 더 크게 반영한다.
- 그 외 분할·특징·C 후보·5-fold 설정은 기본 실험과 똑같다.
- 결과는 `open_binary_balanced`라는 이름으로 요약 표에 함께 저장한다.

**어떻게 보는가**
- 기본 결과와 mon→mon, unmon→unmon, 정확도를 나란히 비교한다.
- 보통 mon→mon은 조금 내려가고 unmon→unmon은 올라가는 교환 관계가 나타난다.
- ROC-AUC가 기본 결과와 비슷하면, 구분 능력 자체가 좋아진 것이 아니라 판정 기준만 옮겨진 것이다.
- 이 결과로 "불균형이 오인의 원인인지"를 확인하며, 균형 가중치만으로 성능 차이를 단정하지 않는다.

In [ ]:
row_bal, _ = run_open_binary(mon, unmon, FEATS, name="open_binary_balanced", class_weight="balanced")
RESULTS.append(row_bal)

base_row = next(r for r in RESULTS if r["experiment"] == "open_binary")
display(pd.DataFrame([base_row, row_bal])[["experiment", "best_C", "test_acc", "mon_as_mon",
                                         "unmon_as_unmon", "precision", "roc_auc"]].round(4))

[open_binary_balanced] best C = 1000 | CV acc = 0.7506 ± 0.0059 | 반복 1616회 | 총 탐색 86.7s
  ⚠️ best C가 탐색 범위의 끝 값 → 범위를 넓혀 확인 필요


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,0.01,0.7376,0.0051,0.4278
1,0.10,0.7464,0.0062,1.1416
2,1.00,0.7472,0.0059,2.1062
3,10.00,0.7489,0.0048,4.7809
4,100.00,0.7504,0.0045,8.0863
5,1000.00,0.7506,0.0059,10.1128


→ test acc 0.7409 (전부 다수 범주로 답할 때 0.6552) | mon→mon 0.7479 | unmon→unmon 0.7275 | ROC-AUC 0.8110


,experiment,best_C,test_acc,mon_as_mon,unmon_as_unmon,precision,roc_auc
0,open_binary,1000,0.7745,0.8905,0.5540,0.7914,0.8118
1,open_binary_balanced,1000,0.7409,0.7479,0.7275,0.8391,0.8110


## 6. 실험 ③ Open-world 다중: 95개 사이트 + unmon(-1) = 96 클래스

- 지표:
  - 정확도 (96 클래스 기준)
  - mon → mon 비율 / unmon → unmon 비율 (사이트 번호는 무시하고 관심 여부만 봤을 때)
  - **mon 사이트 정답 비율**: 실제 관심 사이트 기록 중 사이트 번호까지 정확히 맞힌 비율

In [ ]:
def run_open_multi(mon, unmon, feats, name="open_multi", C_grid=None):
    df = pd.concat([mon, unmon], ignore_index=True)   # unmon label은 이미 -1
    tr, te = df[df.split == "train"], df[df.split == "test"]
    model, info = fit_lr(tr[feats], tr.label, name, C_grid=C_grid)
    pred, t_pred = timed_predict(model, te[feats])
    y = te.label.to_numpy()
    m = openworld_metrics(np.where(y >= 0, 1, -1), np.where(pred >= 0, 1, -1))
    is_mon = y >= 0
    row = dict(experiment=name, trim=int(df.trim.iloc[0]), **info,
               test_acc=accuracy_score(y, pred), mon_site_exact=(pred[is_mon] == y[is_mon]).mean(),
               test_macro_f1=f1_score(y, pred, average="macro"), predict_time_s=t_pred, **m)
    print(f"→ test acc {row['test_acc']:.4f} | mon→mon {m['mon_as_mon']:.4f} | unmon→unmon {m['unmon_as_unmon']:.4f} | mon 사이트 정답 {row['mon_site_exact']:.4f}")
    PREDS[name] = te[["record_id", "dataset", "label"]].assign(pred=pred)
    return row, model

# 가장 오래 걸리는 실험이다. 시간이 부담되면 C_grid=[1, 10, 100, 1000]처럼 작은 C를 빼고 돌린다.
row, model_multi = run_open_multi(mon, unmon, FEATS)
RESULTS.append(row)

pd.DataFrame(RESULTS).to_csv(os.path.join(RESULT_DIR, "lr_summary_partial.csv"), index=False)

[open_multi] best C = 100 | CV acc = 0.6321 ± 0.0041 | 반복 2963회 | 총 탐색 2962.8s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,0.01,0.4249,0.0023,27.6341
1,0.10,0.5401,0.0027,50.1285
2,1.00,0.5959,0.0018,64.0002
3,10.00,0.6204,0.0040,129.2706
4,100.00,0.6321,0.0041,302.0173
5,1000.00,0.6319,0.0043,342.0523


→ test acc 0.6343 | mon→mon 0.7479 | unmon→unmon 0.7810 | mon 사이트 정답 0.5571


## 7. 초반 제거 비교

---

Open-world 이진 분류에서 **원본 · 앞 10개 제거 · 앞 20개 제거**를 같은 분할·같은 설정으로 비교한다.

- 세 조건 모두 5번에서 고른 C(`BEST_C_BINARY`)로 **고정**한다(제안서 6-4 ②의 "동일한 모델 설정").
- 정확도와 두 범주의 정답 개수(TP, TN), ROC-AUC를 함께 본다.
- 제거만으로 수집 편향 해소를 단정하지 않는다. 결과 해석은 아래 7-추가 표와 함께 본다.

In [ ]:
trim_rows = []
for t in TRIMS_TO_COMPARE:
    m_t, _ = load_features("mon", t)
    u_t, _ = load_features(UNMON_NAME, t)
    r, _ = run_open_binary(attach_split(m_t), attach_split(u_t), FEATS,
                           name="open_binary_trimcmp", C_grid=[BEST_C_BINARY])   # 모든 조건에 같은 C
    trim_rows.append(r)
trim_table = pd.DataFrame(trim_rows)[["trim", "best_C", "cv_acc", "cv_std", "test_acc", "roc_auc",
                                      "mon_as_mon", "unmon_as_unmon", "TP", "TN", "FP", "FN"]]
display(trim_table.round(4))
RESULTS.extend(trim_rows)

features_mon_trim0.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim0.csv: (10000, 116), NaN 0, inf 0
[open_binary_trimcmp] best C = 1000 | CV acc = 0.7813 ± 0.0052 | 반복 1515회 | 총 탐색 40.9s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,1000,0.7813,0.0052,8.5851


→ test acc 0.7745 (전부 다수 범주로 답할 때 0.6552) | mon→mon 0.8905 | unmon→unmon 0.5540 | ROC-AUC 0.8118
features_mon_trim10.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim10.csv: (10000, 116), NaN 0, inf 0
[open_binary_trimcmp] best C = 1000 | CV acc = 0.8793 ± 0.0037 | 반복 1233회 | 총 탐색 30.9s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,1000,0.8793,0.0037,6.337


→ test acc 0.8817 (전부 다수 범주로 답할 때 0.6552) | mon→mon 0.9734 | unmon→unmon 0.7075 | ROC-AUC 0.9206
features_mon_trim20.csv: (19000, 116), NaN 0, inf 0
features_unmon10k_trim20.csv: (10000, 116), NaN 0, inf 0
[open_binary_trimcmp] best C = 1000 | CV acc = 0.7710 ± 0.0056 | 반복 1081회 | 총 탐색 33.1s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,1000,0.771,0.0056,7.453


→ test acc 0.7731 (전부 다수 범주로 답할 때 0.6552) | mon→mon 0.8700 | unmon→unmon 0.5890 | ROC-AUC 0.8333


,trim,best_C,cv_acc,cv_std,test_acc,roc_auc,mon_as_mon,unmon_as_unmon,TP,TN,FP,FN
0,0,1000,0.7813,0.0052,0.7745,0.8118,0.8905,0.5540,3384,1108,892,416
1,10,1000,0.8793,0.0037,0.8817,0.9206,0.9734,0.7075,3699,1415,585,101
2,20,1000,0.7710,0.0056,0.7731,0.8333,0.8700,0.5890,3306,1178,822,494


### 7-추가. 초반 패킷 방향 비교 (trim 결과 해석용)

원본 pkl에서 앞 40개 패킷의 **위치별 송신 비율**을 mon과 unmon으로 나눠 비교한다. 특징 CSV에는 위치별 방향이 요약되어 사라지므로 원본을 직접 읽는다.

**확인된 사실**
- 0~6번 패킷은 mon과 unmon의 방향이 완전히 같다.
- **7번 패킷**은 mon에서 거의 항상 송신, unmon에서 거의 항상 수신이다. 이 규칙 하나로 mon/unmon이 거의 완벽히 갈린다.
- 8~13번은 mon에서 거의 항상 수신인데, unmon에서는 송신이 자주 섞인다.
- 20번대 이후에도 송신 비율 차이가 남는다.

**해석**
- 사이트 내용과 무관하게 데이터 묶음(mon / unmon) 단위로 갈리는 차이이므로, 제안서 3-2에서 "검증할 가능성"으로 둔 **수집 흔적**으로 본다.
- 앞 10개를 지우면 `g6_cumul_00`(첫 패킷의 방향)이 원래 10번 패킷의 방향이 되어, 이 차이가 특징 하나로 드러난다. **trim10의 정확도 상승은 이 때문이며, 흔적이 사라졌다는 뜻이 아니다.**
- trim20에도 차이가 남아 있으므로 초반 제거만으로 수집 편향이 해소됐다고 보지 않는다.

원본 pkl을 찾지 못하면 이 셀은 건너뛴다(다른 셀에는 영향 없음).

In [ ]:
def find_raw(fname):
    p = os.path.join(RAW_DIR, fname)
    return p if os.path.exists(p) else None

def first_dirs(records, n=40):
    return np.array([[1 if c > 0 else 0 for c in r[:n]] for r in records])   # 1=송신, 0=수신 (제공 코드와 같은 규칙)

raw_unmon_name = {"unmon10k": "unmon_standard10.pkl", "unmon3k": "unmon_standard10_3000.pkl"}[UNMON_NAME]
p_mon_raw, p_unmon_raw = find_raw("mon_standard.pkl"), find_raw(raw_unmon_name)

if p_mon_raw is None or p_unmon_raw is None:
    print("원본 pkl을 찾지 못해 건너뜀 → 0번 설정의 RAW_DIR을 확인:", RAW_DIR)
else:
    print("원본:", p_mon_raw, "|", p_unmon_raw)
    with open(p_mon_raw, "rb") as f:
        raw_mon = pickle.load(f)
    with open(p_unmon_raw, "rb") as f:
        raw_unmon = pickle.load(f)
    D_mon = first_dirs([s for i in range(950) for s in raw_mon[i]])
    D_unmon = first_dirs(raw_unmon)
    del raw_mon, raw_unmon

    pos_table = pd.DataFrame({"mon_out_ratio": D_mon.mean(0), "unmon_out_ratio": D_unmon.mean(0)})
    pos_table["abs_diff"] = (pos_table.mon_out_ratio - pos_table.unmon_out_ratio).abs()
    pos_table.index.name = "packet_pos"
    display(pos_table.round(3))

    rule_acc = ((D_mon[:, 7] == 1).sum() + (D_unmon[:, 7] == 0).sum()) / (len(D_mon) + len(D_unmon))
    print(f"'7번 패킷이 송신이면 mon' 규칙 하나의 정확도: {rule_acc:.4f}")
    print("7번 패킷이 송신인 기록: mon", int(D_mon[:, 7].sum()), "/", len(D_mon),
          "| unmon", int(D_unmon[:, 7].sum()), "/", len(D_unmon))
    print("8~13번에 송신이 하나라도 있는 기록: mon", int(D_mon[:, 8:14].any(1).sum()), "/", len(D_mon),
          "| unmon", int(D_unmon[:, 8:14].any(1).sum()), "/", len(D_unmon))
    pos_table.to_csv(os.path.join(RESULT_DIR, "lr_early_packet_direction.csv"))

원본: /content/drive/MyDrive/MLB-Project/data/mon_standard.pkl | /content/drive/MyDrive/MLB-Project/data/unmon_standard10.pkl


,mon_out_ratio,unmon_out_ratio,abs_diff
packet_pos,,,
0,0.000,0.000,0.000
1,0.000,0.000,0.000
2,1.000,1.000,0.000
3,0.000,0.000,0.000
4,1.000,1.000,0.000
5,0.000,0.000,0.000
6,1.000,1.000,0.000
7,1.000,0.000,1.000
8,0.000,0.393,0.393


'7번 패킷이 송신이면 mon' 규칙 하나의 정확도: 0.9999
7번 패킷이 송신인 기록: mon 18999 / 19000 | unmon 2 / 10000
8~13번에 송신이 하나라도 있는 기록: mon 4 / 19000 | unmon 8625 / 10000


## 8. 계수 해석
표준화한 특징의 계수라서 크기를 서로 비교할 수 있다. 다만 아래 이유로 **계수는 방향(부호)과 대략적 경향을 보는 참고 자료**로만 쓰고, 그룹별 기여도 결론은 9번·9-추가 그룹 제거 실험으로 판단한다.

- **Open-world 이진**: 계수가 양수면 관심 사이트(mon) 쪽, 음수면 unmon 쪽으로 미는 특징이다.
- **Closed-world**: 95개 클래스별 계수의 절댓값 평균으로 특징·그룹의 영향력을 본다.
- 상관이 높은 특징끼리는 계수가 나눠 갖거나 부호가 뒤집힐 수 있다. 특히 누적 흐름(g6)은 이웃 지점끼리 값이 거의 같아서, 큰 양수와 큰 음수 계수가 번갈아 나오며 그룹 평균 |계수|가 부풀려질 수 있다.
- **계수 크기는 C 선택에 민감하다.** 5번 실험에서 C별 교차검증 정확도는 0.7809~0.7813으로 차이가 0.0005인데, 같은 조건의 표준편차는 0.0052다. 즉 C 후보들이 사실상 동점이라 어느 값이 뽑히는지는 우연에 가깝다. 그런데 C가 커질수록 규제가 약해져 계수의 절대 크기가 크게 늘고, 그룹별 평균 |계수| 순위도 달라진다. 따라서 "어느 그룹이 중요한가"를 계수 크기로 결론 내지 않는다.
- trim0에서 `g6_cumul_00`은 모든 기록에서 같은 값(첫 패킷은 항상 수신)이라 정보가 없다. 아래에서 이런 특징을 출력해 확인한다.

In [ ]:
coef_bin = pd.Series(model_binary[-1].coef_[0], index=FEATS)
top = pd.concat([coef_bin.sort_values(ascending=False).head(10).rename("→ mon"),
                 coef_bin.sort_values().head(10).rename("→ unmon")], axis=1)
print("[Open-world 이진] 계수 상위/하위 10개"); display(top.round(3))

coef_cl = pd.DataFrame(np.abs(model_closed[-1].coef_), columns=FEATS).mean()
group_imp = pd.DataFrame({
    "n_features": {g: len(c) for g, c in GROUPS.items()},
    "closed_mean_abs_coef": {g: coef_cl[c].mean() for g, c in GROUPS.items()},
    "binary_mean_abs_coef": {g: coef_bin[c].abs().mean() for g, c in GROUPS.items()},
})
print("[그룹별 평균 |계수|]"); display(group_imp.round(4))
print("[Closed-world] 평균 |계수| 상위 15개"); display(coef_cl.sort_values(ascending=False).head(15).round(4))

const = [c for c in FEATS if mon[c].nunique() == 1 and unmon[c].nunique() == 1]
print("값이 하나뿐이라 정보가 없는 특징:", const)

[Open-world 이진] 계수 상위/하위 10개


,→ mon,→ unmon
g6_cumul_45,14.341,NaN
g6_cumul_41,11.767,NaN
g6_cumul_20,10.094,NaN
g6_cumul_32,6.285,NaN
g3_bursts_len1_5,6.104,NaN
g6_cumul_27,4.989,NaN
g6_cumul_49,4.710,NaN
g6_cumul_24,4.513,NaN
g6_cumul_25,4.109,NaN
g6_cumul_36,3.865,NaN


[그룹별 평균 |계수|]


,n_features,closed_mean_abs_coef,binary_mean_abs_coef
g1,3,5.7187,2.2937
g2,17,2.5899,0.3988
g3,6,2.7411,4.9945
g4,3,1.1588,0.2813
g5,31,2.0462,0.2520
g6,50,3.7560,3.7393


[Closed-world] 평균 |계수| 상위 15개


,0
g2_out_pos_std,7.0776
g1_n_out,6.8251
g2_out_pos_mean,6.6842
g1_in_ratio,5.4890
g6_cumul_49,5.2121
g6_cumul_06,5.1999
g6_cumul_05,5.0587
g6_cumul_04,5.0459
g6_cumul_08,4.8888
g1_n_total,4.8421


값이 하나뿐이라 정보가 없는 특징: ['g6_cumul_00']


### 8-추가. 오류 분석 (Closed-world)

제안서 6-4 ③의 "잘못 예측한 기록으로 주요 오류 확인"에 해당한다. 최종 평가 기록(사이트당 40개)에서

- 사이트별 정확도: 어떤 사이트가 특히 안 맞는지
- 가장 자주 혼동한 (정답 → 예측) 쌍: 어떤 사이트끼리 헷갈리는지

를 본다. 사이트당 평가 기록이 40개뿐이라 한두 개 차이는 우연일 수 있다.

In [ ]:
pc = PREDS["closed_world"]
site_acc = (pc.label == pc.pred).groupby(pc.label).mean().rename("test_acc")
print("[Closed-world] 정확도가 낮은 사이트 10개"); display(site_acc.sort_values().head(10).round(3))
print("[Closed-world] 정확도가 높은 사이트 5개"); display(site_acc.sort_values().tail(5).round(3))

wrong = pc[pc.label != pc.pred]
pairs = wrong.groupby(["label", "pred"]).size().sort_values(ascending=False).head(15).rename("count").reset_index()
print("[Closed-world] 가장 자주 혼동한 (정답 → 예측) 15쌍"); display(pairs)

site_acc.to_csv(os.path.join(RESULT_DIR, "lr_closed_site_acc.csv"))
pairs.to_csv(os.path.join(RESULT_DIR, "lr_closed_confused_pairs.csv"), index=False)

[Closed-world] 정확도가 낮은 사이트 10개


,test_acc
label,
37,0.250
77,0.275
13,0.325
24,0.350
10,0.350
0,0.375
14,0.400
68,0.425
92,0.425


[Closed-world] 정확도가 높은 사이트 5개


,test_acc
label,
18,0.925
75,0.925
73,0.925
20,0.950
56,0.975


[Closed-world] 가장 자주 혼동한 (정답 → 예측) 15쌍


,label,pred,count
0,13,88,6
1,32,68,6
2,52,31,5
3,37,52,4
4,81,92,4
5,68,25,4
6,77,59,4
7,0,17,4
8,8,81,3
9,69,16,3


## 9. (선택) 특징 그룹 제거 실험

Closed-world에서 110개 전체 조건 + 6개 그룹을 하나씩 뺀 조건 = 7조건을 **같은 분할·같은 C**(4번에서 고른 `BEST_C_CLOSED`)로 다시 학습하고 교차검증 평균 정확도 변화를 본다.
조건마다 C를 다시 고르지 않는다(제안서 6-4 ①의 "같은 데이터 분할과 모델 설정").
제안서는 "대표 모델 하나"로 하기로 했으므로, 팀에서 LR로 정했을 때만 `RUN_ABLATION = True`로 둔다.

In [ ]:
if RUN_ABLATION:
    tr = mon[mon.split == "train"]
    abl = []
    for g in [None] + list(GROUPS):
        feats = FEATS if g is None else [c for c in FEATS if c not in GROUPS[g]]
        _, info = fit_lr(tr[feats], tr.label, f"ablation -{g}", C_grid=[BEST_C_CLOSED])   # C 고정
        abl.append(dict(removed=g or "(없음, 110개)", n_features=len(feats), C=info["best_C"],
                        cv_acc=info["cv_acc"], cv_std=info["cv_std"]))
    abl = pd.DataFrame(abl)
    abl["delta_vs_full"] = abl.cv_acc - abl.cv_acc.iloc[0]
    display(abl.round(4))
    abl.to_csv(os.path.join(RESULT_DIR, "lr_ablation_closed.csv"), index=False)
else:
    print("RUN_ABLATION = False → 건너뜀")

[ablation -None] best C = 100 | CV acc = 0.6626 ± 0.0065 | 반복 1221회 | 총 탐색 390.9s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.6626,0.0065,79.6604


[ablation -g1] best C = 100 | CV acc = 0.6543 ± 0.0068 | 반복 1230회 | 총 탐색 432.1s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.6543,0.0068,82.7512


[ablation -g2] best C = 100 | CV acc = 0.6357 ± 0.0062 | 반복 1355회 | 총 탐색 355.5s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.6357,0.0062,77.1742


... (5 outputs left out)

,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.6036,0.0062,75.2855


[ablation -g6] best C = 100 | CV acc = 0.6011 ± 0.0044 | 반복 605회 | 총 탐색 132.8s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.6011,0.0044,28.6122


,removed,n_features,C,cv_acc,cv_std,delta_vs_full
0,"(없음, 110개)",110,100,0.6626,0.0065,0.0000
1,g1,107,100,0.6543,0.0068,-0.0082
2,g2,93,100,0.6357,0.0062,-0.0269
3,g3,104,100,0.6498,0.0050,-0.0128
4,g4,107,100,0.6295,0.0100,-0.0330
5,g5,79,100,0.6036,0.0062,-0.0590
6,g6,60,100,0.6011,0.0044,-0.0615


### 9-추가. 그룹 제거 실험 (Open-world 이진)

9번과 같은 방식을 이진 분류(mon 1 / unmon -1)에도 적용한다. 5번에서 고른 C로 고정하고, 그룹을 하나씩 빼면서 교차검증 평균 정확도 변화를 본다.

8번 계수는 C 선택에 민감해 크기 비교가 어려우므로, **이진에서의 그룹 기여도는 이 표로 판단한다.**

> 이 셀은 5번 실험에서 만들어지는 `binary_view`와 `BEST_C_BINARY`를 쓴다. 부분 실행할 때는 0~3번과 5번을 먼저 실행한다.

In [ ]:
tr_b = binary_view(mon, unmon)
tr_b = tr_b[tr_b.split == "train"]
abl_b = []
for g in [None] + list(GROUPS):
    feats = FEATS if g is None else [c for c in FEATS if c not in GROUPS[g]]
    _, info = fit_lr(tr_b[feats], tr_b.y, f"ablation_bin -{g}", C_grid=[BEST_C_BINARY])   # C 고정
    abl_b.append(dict(removed=g or "(없음, 110개)", n_features=len(feats),
                      cv_acc=info["cv_acc"], cv_std=info["cv_std"]))
abl_b = pd.DataFrame(abl_b)
abl_b["delta_vs_full"] = abl_b.cv_acc - abl_b.cv_acc.iloc[0]
display(abl_b.round(4))
abl_b.to_csv(os.path.join(RESULT_DIR, "lr_ablation_binary.csv"), index=False)

[ablation_bin -None] best C = 10 | CV acc = 0.7810 ± 0.0042 | 반복 573회 | 총 탐색 26.4s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,10,0.781,0.0042,6.2135


[ablation_bin -g1] best C = 10 | CV acc = 0.7753 ± 0.0038 | 반복 640회 | 총 탐색 24.9s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,10,0.7753,0.0038,5.7091


[ablation_bin -g2] best C = 10 | CV acc = 0.7759 ± 0.0022 | 반복 720회 | 총 탐색 23.4s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,10,0.7759,0.0022,4.503


... (5 outputs left out)

,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,10,0.7619,0.0059,4.5876


[ablation_bin -g6] best C = 10 | CV acc = 0.7781 ± 0.0039 | 반복 147회 | 총 탐색 3.9s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,10,0.7781,0.0039,0.8368


,removed,n_features,cv_acc,cv_std,delta_vs_full
0,"(없음, 110개)",110,0.7810,0.0042,0.0000
1,g1,107,0.7753,0.0038,-0.0058
2,g2,93,0.7759,0.0022,-0.0051
3,g3,104,0.7632,0.0033,-0.0178
4,g4,107,0.7506,0.0037,-0.0304
5,g5,79,0.7619,0.0059,-0.0192
6,g6,60,0.7781,0.0039,-0.0029


### 실험1, 2를 위한 준비

In [ ]:
# 실험 1·2 준비 — 4·5번 셀을 건너뛸 때 필요한 것만 정의한다
def binary_view(mon, unmon):          # 5번 셀과 동일한 정의
    df = pd.concat([mon, unmon], ignore_index=True)
    df["y"] = np.where(df.dataset == "mon", 1, -1)
    return df

# 그룹 제거 실험과 같은 C를 써야 두 결과를 나란히 비교할 수 있다
C_CLOSED = 100     # lr_ablation_closed.csv 와 동일
C_BINARY = 10      # lr_ablation_binary.csv 와 동일
print(f"사용할 C — Closed: {C_CLOSED} / 이진: {C_BINARY}")

사용할 C — Closed: 100 / 이진: 10


### 실험 1. Feature Group별 단독 학습 (Group-only)

각 그룹만으로 학습해 그 그룹 자체의 정보량을 본다. 그룹 외의 조건(모델·분할·C)은 전체 특징 기준과 동일하게 고정한다.

기존 그룹 제거 실험과 함께 읽는다.
- 단독 높음 + 제거 영향 큼 → 핵심이고 대체 불가
- 단독 높음 + 제거 영향 작음 → 유용하지만 다른 그룹과 정보가 중복
- 단독 낮음 + 제거 영향 큼 → 혼자서는 약하지만 다른 그룹을 보완
- 단독 낮음 + 제거 영향 작음 → 기여가 작음

단독 성능이 낮다는 이유만으로 특징을 제거하지 않는다.

In [ ]:
C_CLOSED = globals().get("BEST_C_CLOSED", 100)     # 4번을 건너뛴 경우 이전 전체 실행 값 사용
C_BINARY = globals().get("BEST_C_BINARY", 1000)
print(f"사용할 C — Closed: {C_CLOSED} / 이진: {C_BINARY}")

tr_c = mon[mon.split == "train"]
df_b = binary_view(mon, unmon)
tr_b = df_b[df_b.split == "train"]

rows = []
for task, tr, y, C in (("closed", tr_c, tr_c.label, C_CLOSED),
                       ("binary", tr_b, tr_b.y,     C_BINARY)):
    _, info = fit_lr(tr[FEATS], y, f"grouponly_{task} 전체", C_grid=[C])
    rows.append(dict(task=task, group="전체(110개)", n_features=len(FEATS),
                     cv_acc=info["cv_acc"], cv_std=info["cv_std"]))
    for g, cols in GROUPS.items():
        _, info = fit_lr(tr[cols], y, f"grouponly_{task} {g}만", C_grid=[C])
        rows.append(dict(task=task, group=g, n_features=len(cols),
                         cv_acc=info["cv_acc"], cv_std=info["cv_std"]))

grouponly = pd.DataFrame(rows)

abl = {"closed": pd.read_csv(os.path.join(RESULT_DIR, "lr_ablation_closed.csv")),
       "binary": pd.read_csv(os.path.join(RESULT_DIR, "lr_ablation_binary.csv"))}
for task in ("closed", "binary"):
    t = grouponly[grouponly.task == task].merge(
        abl[task][["removed", "delta_vs_full"]].rename(
            columns={"removed": "group", "delta_vs_full": "제거_시_변화"}),
        on="group", how="left")
    print(f"\n[{task}] 단독 성능 vs 제거 영향")
    display(t[["group", "n_features", "cv_acc", "cv_std", "제거_시_변화"]]
            .sort_values("cv_acc", ascending=False).round(4))

grouponly.to_csv(os.path.join(RESULT_DIR, "lr_grouponly.csv"), index=False)

사용할 C — Closed: 100 / 이진: 1000
[grouponly_closed 전체] best C = 100 | CV acc = 0.6626 ± 0.0065 | 반복 1221회 | 총 탐색 410.1s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.6626,0.0065,84.3437


[grouponly_closed g1만] best C = 100 | CV acc = 0.1486 ± 0.0023 | 반복 219회 | 총 탐색 30.9s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.1486,0.0023,5.8241


[grouponly_closed g2만] best C = 100 | CV acc = 0.3332 ± 0.0075 | 반복 473회 | 총 탐색 78.1s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.3332,0.0075,16.4357


... (22 outputs left out)


[closed] 단독 성능 vs 제거 영향


,group,n_features,cv_acc,cv_std,제거_시_변화
0,전체(110개),110,0.6626,0.0065,NaN
6,g6,50,0.3516,0.0085,-0.0615
2,g2,17,0.3332,0.0075,-0.0269
5,g5,31,0.3095,0.0050,-0.0590
3,g3,6,0.1927,0.0047,-0.0128
1,g1,3,0.1486,0.0023,-0.0082
4,g4,3,0.1476,0.0051,-0.0330



[binary] 단독 성능 vs 제거 영향


,group,n_features,cv_acc,cv_std,제거_시_변화
0,전체(110개),110,0.7813,0.0052,NaN
4,g4,3,0.7268,0.0042,-0.0304
5,g5,31,0.6814,0.0048,-0.0192
6,g6,50,0.6814,0.0025,-0.0029
2,g2,17,0.6663,0.0038,-0.0051
1,g1,3,0.6601,0.0022,-0.0058
3,g3,6,0.6587,0.0038,-0.0178


### 실험 2. 누적 흐름(g6) 표현 해상도 비교

g6의 지점 수만 50개에서 100개로 바꾸고 나머지 특징·분할·C는 고정해 비교한다.
`cumul100_mon_trim0.csv`가 monitored trim0만 있어 Closed-world에서만 수행한다.
g1 + g6 조합은 SVM 보고서의 조건 A·D와 같은 구성이라 모델 간 비교가 가능하다.

In [ ]:
C_CLOSED = globals().get("BEST_C_CLOSED", 100)

c100 = pd.read_csv(os.path.join(FEATURE_DIR, "cumul100_mon_trim0.csv"))
C100 = [c for c in c100.columns if c.startswith("g6_cumul100_")]
print(f"100개 지점 파일: {c100.shape}, 누적 열 {len(C100)}개")

mon100 = mon.merge(c100, on="record_id", how="left", validate="one_to_one")
assert mon100[C100].notna().all().all(), "record_id 결합 실패: 빠진 기록이 있음"

G6_50, G1 = GROUPS["g6"], GROUPS["g1"]
OTHER = [c for c in FEATS if c not in G6_50]          # g1~g5, 60개
CONFIGS = {
    "전체 (g6 50개)":    OTHER + G6_50,
    "전체 (g6 100개)":   OTHER + C100,
    "g1+g6 (g6 50개)":   G1 + G6_50,
    "g1+g6 (g6 100개)":  G1 + C100,
}

tr100 = mon100[mon100.split == "train"]
rows = []
for name, cols in CONFIGS.items():
    _, info = fit_lr(tr100[cols], tr100.label, f"g6res {name}", C_grid=[C_CLOSED])
    rows.append(dict(구성=name, 특징수=len(cols),
                     cv_acc=info["cv_acc"], cv_std=info["cv_std"]))

g6res = pd.DataFrame(rows)
display(g6res.round(4))
g6res.to_csv(os.path.join(RESULT_DIR, "lr_g6_resolution.csv"), index=False)

100개 지점 파일: (19000, 101), 누적 열 100개
[g6res 전체 (g6 50개)] best C = 100 | CV acc = 0.6630 ± 0.0069 | 반복 1288회 | 총 탐색 536.1s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.663,0.0069,109.6754


[g6res 전체 (g6 100개)] best C = 100 | CV acc = 0.6814 ± 0.0090 | 반복 1704회 | 총 탐색 775.9s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.6814,0.009,160.0153


[g6res g1+g6 (g6 50개)] best C = 100 | CV acc = 0.4533 ± 0.0070 | 반복 1810회 | 총 탐색 584.6s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.4533,0.007,139.9743


[g6res g1+g6 (g6 100개)] best C = 100 | CV acc = 0.4937 ± 0.0071 | 반복 2663회 | 총 탐색 990.6s


,C,cv_acc_mean,cv_acc_std,fit_time_per_fold
0,100,0.4937,0.0071,218.0637


,구성,특징수,cv_acc,cv_std
0,전체 (g6 50개),110,0.6630,0.0069
1,전체 (g6 100개),160,0.6814,0.0090
2,g1+g6 (g6 50개),53,0.4533,0.0070
3,g1+g6 (g6 100개),103,0.4937,0.0071


### 실험 5를 위한 준비: 공통 fold · 확률 파일

실험 5(임계값)와 실험 6(앙상블)은 **확률값**이 필요해서, LR을 같은 C로 한 번 더 학습해 확률을 저장한다.

- **공통 fold 파일** `splits/cv_folds_seed42.csv`: 세 모델이 같은 기록을 같은 fold로 나누도록 맞춘 파일. 없으면 여기서 만들고, 있으면 읽기만 한다.
- **OOF 확률** `lr_oof_{과제}.csv`: 학습 데이터를 공통 fold 5개로 나눠, 각 fold를 나머지 4개로 학습한 모델로 예측한 확률. 임계값을 정할 때 쓴다.
- **test 확률** `lr_test_proba_{과제}.csv`: 학습 데이터 전체로 다시 학습한 모델의 평가 데이터 확률. 마지막 평가에만 쓴다.
- C는 위 ①②③에서 고른 값으로 고정한다(다시 찾지 않음).

세 모델의 행을 맞추려고 **record_id 순으로 정렬**해서 다시 학습했기 때문에, 위 ①②③과 숫자가 아주 조금 다르다.
Closed 0.6884 → 0.6861 (3,800개 중 약 9개), 이진 0.7745 → 0.7745, 다중 0.6343 → 0.6338 (5,800개 중 약 3개). 결론은 같다.

In [ ]:
# 실험 5 · (별도 노트북의) 실험 6 공통 설정
from sklearn.model_selection import cross_val_predict

FOLD_PATH      = str(ROOT / "splits" / "cv_folds_seed42.csv")      # 모델팀 공통 fold
LR_C           = {"closed": 100, "binary": 1000, "multi": 100}       # 위 ①②③에서 고른 C
MAX_ITER_PROB  = 5000          # 다중이 3,000회 한도에 가까웠으므로 여유 있게
SKIP_IF_EXISTS = True          # 확률 파일이 이미 있으면 다시 계산하지 않음
TARGET_FPR     = [0.01, 0.05, 0.10, 0.20]

def make_lr_prob(C):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=C, max_iter=MAX_ITER_PROB, random_state=SEED))

def task_frames(task):
    if task == "closed":
        d = mon.copy(); d["y"] = d.label                     # 0~94
    elif task == "binary":
        d = binary_view(mon, unmon)                          # mon 1 / unmon -1
    elif task == "multi":
        d = pd.concat([mon, unmon], ignore_index=True); d["y"] = d.label   # 0~94, unmon -1
    else:
        raise ValueError(task)
    d = d.sort_values("record_id").reset_index(drop=True)   # 모델팀 누구나 같은 행 순서
    return (d[d.split == "train"].reset_index(drop=True),
            d[d.split == "test"].reset_index(drop=True))

In [ ]:
if os.path.exists(FOLD_PATH):
    folds = pd.read_csv(FOLD_PATH)
    print("기존 공통 fold 파일 사용:", FOLD_PATH)
else:
    parts = []
    for task in ("closed", "binary", "multi"):
        tr, _ = task_frames(task)
        f = np.empty(len(tr), dtype=int)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        for k, (_, va) in enumerate(skf.split(tr, tr.y)):
            f[va] = k
        parts.append(pd.DataFrame({"task": task, "record_id": tr.record_id, "fold": f}))
    folds = pd.concat(parts, ignore_index=True)
    folds.to_csv(FOLD_PATH, index=False)
    print("공통 fold 파일 새로 저장:", FOLD_PATH)
display(folds.groupby(["task", "fold"]).size().unstack())

공통 fold 파일 새로 저장: /content/drive/MyDrive/MLB-Project/results/cv_folds_seed42.csv


fold,0,1,2,3,4
task,,,,,
binary,4640,4640,4640,4640,4640
closed,3040,3040,3040,3040,3040
multi,4640,4640,4640,4640,4640


In [ ]:
def prob_paths(task):
    return (os.path.join(RESULT_DIR, f"lr_oof_{task}.csv"),
            os.path.join(RESULT_DIR, f"lr_test_proba_{task}.csv"))

def oof_and_test(task):
    p_oof, p_te = prob_paths(task)
    if SKIP_IF_EXISTS and os.path.exists(p_oof) and os.path.exists(p_te):
        print(f"[{task}] 이미 저장된 파일 사용 (다시 계산하려면 SKIP_IF_EXISTS = False)")
        return pd.read_csv(p_oof), pd.read_csv(p_te)

    tr, te = task_frames(task)
    fold = folds[folds.task == task].set_index("record_id").fold.loc[tr.record_id].to_numpy()
    splits = [(np.where(fold != k)[0], np.where(fold == k)[0]) for k in range(5)]
    model = make_lr_prob(LR_C[task])

    t0 = time.time()
    oof = cross_val_predict(model, tr[FEATS], tr.y, cv=splits, method="predict_proba", n_jobs=N_JOBS)
    t_oof = time.time() - t0
    t0 = time.time(); model.fit(tr[FEATS], tr.y); t_fit = time.time() - t0
    te_p = model.predict_proba(te[FEATS])

    cols = [f"p_{c}" for c in model.classes_]
    out_oof = pd.concat([tr[["record_id", "y"]].assign(fold=fold), pd.DataFrame(oof, columns=cols)], axis=1)
    out_te  = pd.concat([te[["record_id", "y"]], pd.DataFrame(te_p, columns=cols)], axis=1)
    out_oof.to_csv(p_oof, index=False)
    out_te.to_csv(p_te, index=False)

    n_iter = int(np.max(model[-1].n_iter_))
    print(f"[{task}] C={LR_C[task]} | OOF {t_oof:.0f}s, 재학습 {t_fit:.0f}s, 반복 {n_iter}회 → 저장 완료")
    if n_iter >= MAX_ITER_PROB:
        print(f"  [경고] 반복 한도({MAX_ITER_PROB})에 도달 — MAX_ITER_PROB를 늘려 다시 실행")
    return out_oof, out_te

def prob_summary(task, oof, te):
    cols = [c for c in oof.columns if c.startswith("p_")]
    classes = np.array([int(c[2:]) for c in cols])
    row = {"task": task}
    for name, d in (("OOF", oof), ("test", te)):
        pred = classes[d[cols].to_numpy().argmax(1)]
        y = d.y.to_numpy()
        row[f"{name} 정확도"] = (pred == y).mean()
        if task == "multi":
            is_mon = y >= 0
            row[f"{name} mon 사이트 정답"] = (pred[is_mon] == y[is_mon]).mean()
    return row

PROB = {}
for task in ("closed", "binary"):
    PROB[task] = oof_and_test(task)
display(pd.DataFrame([prob_summary(t, *PROB[t]) for t in PROB]).round(4))

[closed] C=100 | OOF 277s, 재학습 221s, 반복 1288회 → 저장 완료
[binary] C=1000 | OOF 27s, 재학습 17s, 반복 1359회 → 저장 완료


,task,OOF 정확도,test 정확도
0,closed,0.663,0.6861
1,binary,0.781,0.7745


다중 분류는 가장 오래 걸려서(약 25분) 셀을 따로 뒀다.

In [ ]:
PROB["multi"] = oof_and_test("multi")
display(pd.DataFrame([prob_summary(t, *PROB[t]) for t in PROB]).round(4))

[multi] C=100 | OOF 858s, 재학습 617s, 반복 2677회 → 저장 완료


,task,OOF 정확도,test 정확도,OOF mon 사이트 정답,test mon 사이트 정답
0,closed,0.6630,0.6861,NaN,NaN
1,binary,0.7810,0.7745,NaN,NaN
2,multi,0.6321,0.6338,0.5526,0.5561


### 실험 5. (보조) Open-world 이진 임계값 분석

- 정확도 하나로는 안 보이는 **탐지율(TPR)과 오탐률(FPR)의 균형**을 본다.
- 임계값은 **검증(OOF) 데이터에서만** 정하고, test에는 그 값을 그대로 적용한다. test를 보고 다시 조정하지 않는다.
- 목표 오탐률은 0.01·0.05·0.10·0.20 네 가지로 계산한다.
- 조교님 안내에 따라 이진 분류는 보조 결과다. 수집 흔적의 영향을 받으므로 모델 간 상대 비교로만 해석한다.

In [ ]:
def threshold_at_fpr(y, p, target):
    """검증 데이터의 unmon 점수에서 오탐률이 target 이하가 되는 임계값. p > t 이면 mon으로 판정."""
    return float(np.quantile(p[y == -1], 1 - target, method="higher"))

def rates(y, p, t):
    pred = p > t
    tp = int((pred & (y == 1)).sum());  fp = int((pred & (y == -1)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == -1)).sum())
    return dict(TPR=tp / (tp + fn), FPR=fp / (fp + tn),
                precision=tp / (tp + fp) if tp + fp else np.nan, acc=(tp + tn) / len(y))

def threshold_table(y_v, p_v, y_t, p_t):
    cands = [("기본 (p>0.5)", 0.5)] + [(f"검증 FPR≤{f:.2f}", threshold_at_fpr(y_v, p_v, f)) for f in TARGET_FPR]
    rows = []
    for name, t in cands:
        v, s = rates(y_v, p_v, t), rates(y_t, p_t, t)
        rows.append({"기준": name, "임계값": t,
                     "검증 TPR": v["TPR"], "검증 FPR": v["FPR"], "검증 Precision": v["precision"],
                     "test TPR": s["TPR"], "test FPR": s["FPR"], "test Precision": s["precision"],
                     "test 정확도": s["acc"]})
    return pd.DataFrame(rows)

oof_b, te_b = PROB["binary"]
y_v, p_v = oof_b.y.to_numpy(), oof_b["p_1"].to_numpy()       # p_1 = mon일 확률
y_t, p_t = te_b.y.to_numpy(), te_b["p_1"].to_numpy()

thr = threshold_table(y_v, p_v, y_t, p_t)
display(thr.round(4))
print(f"ROC-AUC  검증 {roc_auc_score(y_v, p_v):.4f} | test {roc_auc_score(y_t, p_t):.4f}")
print(f"PR-AUC   검증 {average_precision_score(y_v, p_v, pos_label=1):.4f} | "
      f"test {average_precision_score(y_t, p_t, pos_label=1):.4f}")
thr.to_csv(os.path.join(RESULT_DIR, "lr_threshold_binary.csv"), index=False)

,기준,임계값,검증 TPR,검증 FPR,검증 Precision,test TPR,test FPR,test Precision,test 정확도
0,기본 (p>0.5),0.5000,0.8978,0.4408,0.7947,0.8908,0.4465,0.7913,0.7745
1,검증 FPR≤0.01,0.9671,0.0730,0.0099,0.9335,0.0755,0.0075,0.9503,0.3917
2,검증 FPR≤0.05,0.8929,0.2760,0.0499,0.9131,0.2826,0.0530,0.9102,0.5117
3,검증 FPR≤0.10,0.8249,0.4538,0.0999,0.8962,0.4555,0.1075,0.8895,0.6062
4,검증 FPR≤0.20,0.7144,0.6697,0.1999,0.8642,0.6676,0.2095,0.8583,0.7100


ROC-AUC  검증 0.8194 | test 0.8118
PR-AUC   검증 0.8734 | test 0.8703


### 실험 6. 앙상블 → 별도 노트북

세 모델의 확률을 합치는 작업이라 `MLB_Experiment6_Ensemble.ipynb`로 분리했다. 이 노트북은 위에서 만든 LR 확률 파일과 공통 fold 파일만 남긴다.

## 10. 결과 저장

In [ ]:
if not RESULTS:
    print("[경고] RESULTS가 비어 있음 → 4~7번 실험을 돌리지 않은 부분 실행 상태.\n   기존 저장 파일을 지키기 위해 저장을 건너뜀. 전체를 처음부터 실행한 뒤 다시 저장할 것.")
else:
    summary = pd.DataFrame(RESULTS)
    cols = ["experiment", "trim", "best_C", "cv_acc", "cv_std", "test_acc", "majority_baseline", "mon_as_mon", "unmon_as_unmon",
            "mon_site_exact", "precision", "fpr", "roc_auc", "pr_auc", "test_macro_f1", "TP", "FN", "TN", "FP",
            "n_iter", "search_time_s", "refit_time_s", "predict_time_s"]
    summary = summary[[c for c in cols if c in summary.columns]]
    display(summary.round(4))

    summary.to_csv(os.path.join(RESULT_DIR, "lr_summary.csv"), index=False)
    pd.concat(CV_TABLES.values(), ignore_index=True).to_csv(os.path.join(RESULT_DIR, "lr_cv_tables.csv"), index=False)   # C별 교차검증 표
    for k, v in PREDS.items():
        v.to_csv(os.path.join(RESULT_DIR, f"lr_pred_{k}.csv"), index=False)   # 잘못 예측한 기록 확인용
    group_imp.to_csv(os.path.join(RESULT_DIR, "lr_group_coef.csv"))
    print("저장 완료:", sorted(os.listdir(RESULT_DIR)))

,experiment,trim,best_C,cv_acc,cv_std,test_acc,majority_baseline,mon_as_mon,unmon_as_unmon,mon_site_exact,precision,fpr,roc_auc,pr_auc,test_macro_f1,TP,FN,TN,FP,n_iter,search_time_s,refit_time_s,predict_time_s
0,closed_world,0,100.0,0.6626,0.0065,0.6884,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.6842,NaN,NaN,NaN,NaN,1221,1042.4820,178.4540,0.0308
1,closed_world_default_C1,0,1.0,0.6239,0.0054,0.6413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.6317,NaN,NaN,NaN,NaN,334,112.6757,52.4582,0.0220
2,open_binary,0,1000.0,0.7813,0.0052,0.7745,0.6552,0.8905,0.5540,NaN,0.7914,0.4460,0.8118,0.8705,NaN,3384.0,416.0,1108.0,892.0,1515,71.7855,16.1082,0.0188
3,open_binary_balanced,0,1000.0,0.7506,0.0059,0.7409,0.6552,0.7479,0.7275,NaN,0.8391,0.2725,0.8110,0.8738,NaN,2842.0,958.0,1455.0,545.0,1616,86.6856,18.5358,0.0285
4,open_multi,0,100.0,0.6321,0.0041,0.6343,NaN,0.7479,0.7810,0.5571,0.8665,0.2190,NaN,NaN,0.5796,2842.0,958.0,1562.0,438.0,2963,2962.7620,654.2220,0.0241
5,open_binary_trimcmp,0,1000.0,0.7813,0.0052,0.7745,0.6552,0.8905,0.5540,NaN,0.7914,0.4460,0.8118,0.8705,NaN,3384.0,416.0,1108.0,892.0,1515,40.8524,15.9802,0.0087
6,open_binary_trimcmp,10,1000.0,0.8793,0.0037,0.8817,0.6552,0.9734,0.7075,NaN,0.8634,0.2925,0.9206,0.9388,NaN,3699.0,101.0,1415.0,585.0,1233,30.8898,13.5435,0.0067
7,open_binary_trimcmp,20,1000.0,0.7710,0.0056,0.7731,0.6552,0.8700,0.5890,NaN,0.8009,0.4110,0.8333,0.8940,NaN,3306.0,494.0,1178.0,822.0,1081,33.0566,12.3189,0.0159


저장 완료: ['lr_ablation_closed.csv', 'lr_closed_confused_pairs.csv', 'lr_closed_site_acc.csv', 'lr_cv_tables.csv', 'lr_early_packet_direction.csv', 'lr_group_coef.csv', 'lr_pred_closed_world.csv', 'lr_pred_closed_world_default_C1.csv', 'lr_pred_open_binary_balanced_trim0.csv', 'lr_pred_open_binary_trim0.csv', 'lr_pred_open_binary_trimcmp_trim0.csv', 'lr_pred_open_binary_trimcmp_trim10.csv', 'lr_pred_open_binary_trimcmp_trim20.csv', 'lr_pred_open_multi.csv', 'lr_summary.csv', 'lr_summary_partial.csv']
